In [1]:
import ccxt
import pandas as pd
import time
from datetime import datetime, timedelta
import os

print("=== MEMULAI MINING DATA INDODAX (TIMEFRAME 1 MENIT - 5 TAHUN) ===")

# 1. Konfigurasi
exchange = ccxt.indodax({
    'enableRateLimit': True,
})
symbol = 'BTC/IDR'
timeframe = '1m'
limit = 1000
csv_filename = 'btc_idr_1m_5years_brute.csv'

# 2. Setup Waktu (Mundur 5 Tahun dari Sekarang)
now = datetime.now()
start_date = now - timedelta(days=5 * 365)
since = int(start_date.timestamp() * 1000)
end_time = int(now.timestamp() * 1000)

print(f"Target Start : {start_date.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Target End   : {now.strftime('%Y-%m-%d %H:%M:%S')}")

# 3. Looping Tarik Data
total_fetched = 0

if not os.path.exists(csv_filename):
    pd.DataFrame(columns=['timestamp', 'open', 'high', 'low', 'close', 'volume']).to_csv(csv_filename, index=False)

try:
    while since < end_time:
        # Tarik data
        bars = exchange.fetch_ohlcv(symbol, timeframe, since, limit)
        
        if len(bars) == 0:
            print("❌ Server Indodax menolak ngasih data 1 menit di tanggal ini (Terlalu lama).")
            break
            
        df = pd.DataFrame(bars, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
        df.to_csv(csv_filename, mode='a', header=False, index=False)
        
        total_fetched += len(bars)
        
        # WAJIB 60000 (1 Menit) biar datanya nggak bolong-bolong!
        last_timestamp = bars[-1][0]
        since = last_timestamp + 60000 
        
        last_time_str = pd.to_datetime(last_timestamp, unit='ms').strftime('%Y-%m-%d %H:%M:%S')
        print(f"✅ Berhasil menarik {len(bars)} candle. Posisi waktu saat ini: {last_time_str} | Total: {total_fetched}")
        
        time.sleep(1.5)

except Exception as e:
    print(f"\n❌ Terjadi Error API: {e}")
    print("Mentok bro! Indodax nggak ngijinin narik data 1 menit dari tahun sejauh itu.")

print(f"\n🎉 Selesai/Berhenti! Total baris terkumpul: {total_fetched} di {csv_filename}")

=== MEMULAI MINING DATA INDODAX (TIMEFRAME 1 MENIT - 5 TAHUN) ===
Target Start : 2021-06-01 21:56:12
Target End   : 2026-05-31 21:56:12

❌ Terjadi Error API: indodax GET https://indodax.com/tradingview/history_v2?to=1780239372&tf=1&symbol=btcidr&from=1622559372
Mentok bro! Indodax nggak ngijinin narik data 1 menit dari tahun sejauh itu.

🎉 Selesai/Berhenti! Total baris terkumpul: 0 di btc_idr_1m_5years_brute.csv
